In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [2]:


train = np.load("../data/processed/TRAIN_30.npz", allow_pickle=True)
X_train = train["X"][::20]
y_train = train["y"][::20]

##### TODO 
##do uczenia są tylko przesunięcia 
## w full nie musimy tego robić
## tutaj przerzedzamy co 10
full = np.load("../data/processed/FULL_30.npz", allow_pickle=True)
X_full  = full["X"][::150]
K0_full = full["K0"][::150]


In [3]:
cols_21 = np.setdiff1d(
    np.arange(30),
    [2,3,5,7,12,18,21,24,27,28]
)
cols_12 = [0,2,4,5,6,11,12,15,17,23,24,29]

In [4]:
feature_sets = {
    "30": np.arange(30),
    "21": cols_21,
    "12": cols_12,
}

In [5]:
summary_rows = []
mean_rows = []
chi_rows = []

In [6]:
def build_model(base_model, calibrated=False):
    pipe = Pipeline([
        # ("scaler", StandardScaler()),
        ("clf", base_model)
    ])
    if not calibrated:
        return pipe

    return CalibratedClassifierCV(pipe, method="sigmoid", cv=3)

In [7]:
def run_supervised_pipeline(model, X_train, y_train, X_full):
    model.fit(X_train, y_train)
    train_score = model.score(X_train, y_train)

    if hasattr(model, "predict_proba"):
        P_A = model.predict_proba(X_full)[:, 1]
    else:
        scores = model.decision_function(X_full)
        P_A = 1.0 / (1.0 + np.exp(-scores))

    return train_score, P_A

In [8]:
def stat_mean(x):
    return np.mean(x)



def stat_susceptibility(x):
    mu = np.mean(x)

    if abs(mu) < 1e-12:
        return 0.0

    return np.var(x, ddof=1) #######/ mu

    
    # def stat_susceptibility(x):
#     # Mathematica: Variance[...] / Mean[...] where Variance uses ddof=1
#     return np.var(x, ddof=1) / np.mean(x)

In [9]:
def block_jackknife(data, stat_func, B):
    """
    Block jackknife with B blocks.
      - truncate data to m*B elements
      - reshape into B blocks of size m
      - compute full estimator EX on truncated data
      - leave one block out at a time, compute stat on remaining B-1 blocks
      - bias = (B-1) * (mean(dats) - EX)          [jack.pdf eq 3.2.1]
      - theta_jack = EX - bias                      [bias-corrected estimate]
      - var = Variance(dats) * (B-1)^2 / B         [Mathematica Variance = ddof=1]
      - se = sqrt(var)

    Returns: (theta_hat, theta_jack, se) for this specific B,
             or None if block size < 1.
    """
    n = len(data)
    m = n // B  # block size (integer division, matches Mathematica Quotient)

    if m < 1:
        return None

    # Truncate to m*B  (matches: data[[1 ;; Quotient[len,B]*B]])
    data = data[:m * B]

    # Reshape into B blocks  (matches: Table[data[[m*i+1 ;; m*(i+1)]], {i,0,B-1}])
    blocks = data.reshape(B, m)

    # Full estimator on truncated data  (matches: EX = stat_func(data))
    theta_hat = stat_func(data)

    # Leave-one-block-out replicates
    # Matches: rands = Table[DeleteCases[Range[B], i], {i, B}]
    #          dats  = Table[stat(Flatten[blocks[rands[i]]]), {i, B}]
    dats = np.array([
        stat_func(np.concatenate([blocks[:i], blocks[i+1:]]).ravel())
        for i in range(B)
    ])

    # Bias: (B-1) * (mean(dats) - EX)  [jack.pdf 3.2.1, Mathematica bias line]
    bias = (B - 1) * (np.mean(dats) - theta_hat)

    # Bias-corrected estimator: EXUn = EX - bias
    theta_jack = theta_hat - bias

    # Variance of replications using ddof=1  (Mathematica Variance = sample variance)
    # var = Variance[dats] * (B-1)^2 / B
    var = np.var(dats, ddof=1) * (B - 1)**2 / B
    se = np.sqrt(var)

    return theta_hat, theta_jack, se



In [10]:
# =============================================================================
# JACKKNIFE OVER K0 — returns FULL TABLE for every B, matching Mathematica output
# =============================================================================

def jackknife_over_K(P_A, K0, stat_func, B_range=range(2, 101)):
    """
    For each unique K0 value, run block_jackknife for every B in B_range.

    Returns a dict:  {k: DataFrame with columns [B, theta_hat, theta_jack, se]}

    This matches the Mathematica Table[..., {ildanych, 2, 100, 1}] output exactly.
    You can then inspect SE vs B plots to find the plateau, rather than
    artificially picking the maximum SE.
    """
    df_input = pd.DataFrame({"K0": K0, "P_A": P_A})
    results = {}

    for k, group in df_input.groupby("K0"):
        data = group["P_A"].values
        # print(k, len(data))

        if len(data) < 2:
            continue

        rows = []
        for B in B_range:
            out = block_jackknife(data, stat_func, B)
            if out is None:
                continue
            theta_hat, theta_jack, se = out
            rows.append({
                "B":          B,
                "theta_hat":  theta_hat,   # EX  in Mathematica
                "theta_jack": theta_jack,  # EXUn in Mathematica
                "se":         se,          # Smse in Mathematica
            })

        results[k] = pd.DataFrame(rows)
    return results

In [11]:
def summarise_jackknife(results_dict):

    K_vals, theta_arr, se_arr = [], [], []

    for k in sorted(results_dict.keys()):
        df = results_dict[k]

        if df.empty:
            continue

        df_valid = df.dropna(subset=["se"])

        if df_valid.empty:
            print(f"All SE are NaN for K0={k}")
            continue

        row = df_valid.loc[df_valid["se"].idxmax()]

        K_vals.append(k)
        theta_arr.append(row["theta_jack"])
        se_arr.append(row["se"])

    return (
        np.array(K_vals),
        np.array(theta_arr),
        np.array(se_arr),
    )



def plot_se_vs_B(results_dict, stat_label, target_Ks=None, ncols=5):
    """
    Plot SE vs B for each K value.
    Use this to verify that the SE has plateaued before trusting the summary.
    All K values are shown, arranged in a grid with ncols columns.
    """
    ks = sorted(results_dict.keys())
    if target_Ks is not None:
        ks = [k for k in ks if k in target_Ks]

    n = len(ks)
    if n == 0:
        return

    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

    for idx, k in enumerate(ks):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]
        df = results_dict[k]
        ax.plot(df["B"], df["se"], lw=1.5)
        ax.set_title(f"K0 = {k:.3f}")
        ax.set_xlabel("B (number of blocks)")
        ax.set_ylabel(f"SE of {stat_label}")
        ax.grid(True, alpha=0.3)

    # Hide any leftover empty subplots
    for idx in range(n, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    fig.suptitle(f"SE vs B — {stat_label} (should plateau)", y=1.02)
    plt.tight_layout()

In [12]:
from scipy.interpolate import interp1d
from scipy.optimize import brentq


def find_k0_crit(K, mean, mean_err):
    """
    Find K where mean(K)=0.5
    and estimate uncertainty via error propagation.
    """

    idx_close = np.argmin(np.abs(mean-0.5))
    k_close = K[idx_close]
    
    f = interp1d(
        K,
        mean - 0.5,
        kind="linear",
        bounds_error=False
    )

    idx = np.where(
        (mean[:-1] - 0.5) *
        (mean[1:]  - 0.5) <= 0
    )[0]

    if len(idx) == 0:
        return np.nan, np.nan, np.nan

    i = idx[0]

    kcrit = brentq(
        lambda x: f(x),
        K[i],
        K[i + 1]
    )

    slope = (
        mean[i + 1] - mean[i]
    ) / (
        K[i + 1] - K[i]
    )

    sigma_p = np.interp(
        kcrit,
        [K[i], K[i + 1]],
        [mean_err[i], mean_err[i + 1]]
    )
    if abs(slope) < 1e-12:
        return kcrit, np.nan, k_close
    
    sigma_k = sigma_p / abs(slope)

    return kcrit, sigma_k, k_close


In [13]:
def timed(func, *args, **kwargs):
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    return result, elapsed

In [14]:
# =============================================================================
# MAIN
# =============================================================================

neighbors = list(range(1, 52, 2)) + [61, 81, 101, 151, 201]

algorithms = [
    "auto",
    "kd_tree",
    "ball_tree",
    "brute",
]

models = {}

for algorithm in algorithms:
    for k in neighbors:
        name = f"KNN | {algorithm} | k={k}"

        models[name] = KNeighborsClassifier(
            n_neighbors=k,
            algorithm=algorithm,
            n_jobs=-1,
        )

curves = {}

for feature_name, cols in feature_sets.items():

    X_train_fs = X_train[:, cols]
    X_full_fs  = X_full[:, cols]

    curves[feature_name] = {}

    print(
        f"\n{'='*70}"
        f"\nFEATURE SET: {feature_name}"
        f" ({len(cols)} features)"
        f"\n{'='*70}"
    )

    for name, base_model in models.items():

        print(f"\n{'='*50}\nModel: {name}\n{'='*50}")
        curves[feature_name][name] = {}

        for calibrated in [False]:

            label = "calibrated" if calibrated else "uncalibrated"

            model = build_model(base_model, calibrated=calibrated)

            (train_score, P_A), model_time = timed(
                run_supervised_pipeline,
                model,
                X_train_fs,
                y_train,
                X_full_fs,
            )

            mean_results, mean_jack_time = timed(
                jackknife_over_K,
                P_A,
                K0_full,
                stat_mean,
            )

            chi_results, chi_jack_time = timed(
                jackknife_over_K,
                P_A,
                K0_full,
                stat_susceptibility,
            )

            print(f"  K0 values found: {len(mean_results)}")

            (K0_mean, mean_jack, mean_se), summary_mean_time = timed(
                summarise_jackknife,
                mean_results,
            )

            (K0_chi, chi_jack, chi_se), summary_chi_time = timed(
                summarise_jackknife,
                chi_results,
            )

            (kcrit, kcrit_err, k_close), time_k0_crit = timed(
                find_k0_crit,
                K0_mean,
                mean_jack,
                mean_se,
            )

            total_time = (
                model_time
                + mean_jack_time
                + chi_jack_time
                + summary_mean_time
                + summary_chi_time
                + time_k0_crit
            )

            print(
                f"{label:12s} | "
                f"train_acc={train_score:.4f} | "
                f"total_time={total_time:.3f}"
            )

            curves[feature_name][name][label] = {
                "train_acc": train_score,
                "runtime_sec": total_time,
                "K0": K0_mean,
                "mean": mean_jack,
                "mean_se": mean_se,
                "chi": chi_jack,
                "chi_se": chi_se,
                "Delta_crit": kcrit,
                "Delta_crit_err": kcrit_err,
                "Delta_close": k_close,
                "_mean_results": mean_results,
                "_chi_results": chi_results,
            }

            summary_rows.append({
                "feature_set": feature_name,
                "n_features": len(cols),

                "model": "KNN",
                "algorithm": base_model.algorithm,
                "n_neighbors": base_model.n_neighbors,

                "calibration": label,
                "train_acc": train_score,

                "K0_crit": kcrit,
                "K0_crit_err": kcrit_err,
                "K0_close": k_close,
                "N_K0": len(K0_mean),

                "runtime_model": round(model_time, 3),
                "runtime_jackknife_mean": round(mean_jack_time, 3),
                "runtime_jackknife_chi": round(chi_jack_time, 3),
                "summary_mean_time": round(summary_mean_time, 4),
                "summary_chi_time": round(summary_chi_time, 4),
                "runtime_K0_crit": round(time_k0_crit, 5),
                "runtime_total": round(total_time, 3),
            })

            for k, m, err in zip(K0_mean, mean_jack, mean_se):
                mean_rows.append({
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": "KNN",
                    "algorithm": base_model.algorithm,
                    "n_neighbors": base_model.n_neighbors,
                    "calibration": label,
                    "K0": k,
                    "mean": m,
                    "mean_err": err,
                })

            for k, c, err in zip(K0_chi, chi_jack, chi_se):
                chi_rows.append({
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": "KNN",
                    "algorithm": base_model.algorithm,
                    "n_neighbors": base_model.n_neighbors,
                    "calibration": label,
                    "K0": k,
                    "chi": c,
                    "chi_err": err,
                })


FEATURE SET: 30 (30 features)

Model: KNN | auto | k=1
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=24.825

Model: KNN | auto | k=3
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=24.769

Model: KNN | auto | k=5
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=24.086

Model: KNN | auto | k=7
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=23.953

Model: KNN | auto | k=9
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=25.668

Model: KNN | auto | k=11
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=24.893

Model: KNN | auto | k=13
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=25.565

Model: KNN | auto | k=15
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=24.970

Model: KNN | auto | k=17
  K0 values found: 20
uncalibrated | train_acc=1.0000 | total_time=25.495

Model: KNN | auto | k=19
  K0 values found: 20
uncalibrated | train_acc=1

In [ ]:
import os

os.makedirs("results", exist_ok=True)

pd.DataFrame(summary_rows).to_csv(
    "results/AC_KNN_COMPARISON_summary.csv",
    index=False
)

pd.DataFrame(mean_rows).to_csv(
    "results/AC_KNN_COMPARISON_curve_meanKNN.csv",
    index=False
)

pd.DataFrame(chi_rows).to_csv(
    "results/AC_KNN_COMPARISON_curve_chiKNN.csv",
    index=False
)

In [ ]:
# import os

# print("Finished successfully. Shutting down...")
# os.system("shutdown -h now")

In [ ]:
# # =============================================================================
# # DIAGNOSTIC: SE vs B plots (mirrors the Mathematica table inspection)
# # =============================================================================

# for name in curves:
#     for label in curves[name]:
#         data = curves[name][label]
#         plot_se_vs_B(data["_mean_results"], stat_label="Mean P_A" )
#         plot_se_vs_B(data["_chi_results"],  stat_label="Chi (Var/Mean)")

# # =============================================================================
# # MAIN PLOTS — fill_between style
# # =============================================================================

# # for name in curves:
# #     for label in curves[name]:
# #         data = curves[name][label]
# #         K = data["K"]

# #         # Mean P_A
# #         plt.figure(figsize=(8, 6))
# #         plt.title(f"{name} ({label}) — Mean P_A with Jackknife Error")
# #         plt.plot(K, data["mean"])
# #         plt.fill_between(
# #             K,
# #             data["mean"] - data["mean_se"],
# #             data["mean"] + data["mean_se"],
# #             alpha=0.7,
# #             color="red",
# #         )
# #         plt.xlabel("K0")
# #         plt.ylabel("Mean P_A")

# #         # Susceptibility
# #         plt.figure(figsize=(8, 6))
# #         plt.title(f"{name} ({label}) — Susceptibility with Jackknife Error")
# #         plt.plot(K, data["chi"])
# #         plt.fill_between(
# #             K,
# #             data["chi"] - data["chi_se"],
# #             data["chi"] + data["chi_se"],
# #             alpha=0.7,
# #             color="red",
# #         )
# #         plt.xlabel("K0")
# #         plt.ylabel("Chi (Variance / Mean)")

# # =============================================================================
# # ERRORBAR PLOTS
# # =============================================================================

# for name in curves:
#     for label in curves[name]:
#         data = curves[name][label]
#         K = data["K"]

#         plt.figure(figsize=(8, 6))
#         plt.errorbar(K, data["mean"], yerr=data["mean_se"], fmt='o', capsize=4)
#         plt.xlabel("K0")
#         plt.ylabel("Mean P_A")
#         plt.title(f"{name} ({label}) — Mean P_A with plateau jackknife error")

#         plt.figure(figsize=(8, 6))
#         plt.errorbar(K, data["chi"], yerr=data["chi_se"], fmt='o', capsize=3)
#         plt.xlabel("K0")
#         plt.ylabel("Chi (Variance / Mean)")
#         plt.title(f"{name} ({label}) — Susceptibility with plateau jackknife error")

# plt.show()

# import os
# os.makedirs("plots", exist_ok=True)
# def plot_se_vs_B(results_dict, stat_label, target_Ks=None, ncols=5, save_path=None):
#     ks = sorted(results_dict.keys())
#     if target_Ks is not None:
#         ks = [k for k in ks if k in target_Ks]

#     n = len(ks)
#     if n == 0:
#         return

#     nrows = int(np.ceil(n / ncols))
#     fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

#     for idx, k in enumerate(ks):
#         row, col = divmod(idx, ncols)
#         ax = axes[row][col]
#         df = results_dict[k]
#         ax.plot(df["B"], df["se"], lw=1.5)
#         ax.set_title(f"K0 = {k:.3f}")
#         ax.set_xlabel("B (number of blocks)")
#         ax.set_ylabel(f"SE of {stat_label}")
#         ax.grid(True, alpha=0.3)

#     for idx in range(n, nrows * ncols):
#         row, col = divmod(idx, ncols)
#         axes[row][col].set_visible(False)

#     fig.suptitle(f"SE vs B — {stat_label} (should plateau)", y=1.02)
#     plt.tight_layout()

#     # ✅ THIS is the only new part
#     if save_path is not None:
#         fig.savefig(save_path, dpi=300, bbox_inches="tight")

#     return fig

In [ ]:
# for name in curves:
#     for label in curves[name]:
#         data = curves[name][label]

#         plot_se_vs_B(
#             data["_mean_results"],
#             stat_label="Mean P_A",
#             save_path=f"plots/{name}_{label}_mean_se_vs_B.png"
#         )

#         plot_se_vs_B(
#             data["_chi_results"],
#             stat_label="Chi (Var/Mean)",
#             save_path=f"plots/{name}_{label}_chi_se_vs_B.png"
#         )